preuba de funciones

In [1]:
import os
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np

In [2]:
current_file = Path(os.getcwd())
# Go up until you find the project root (where "src" exists)
for parent in current_file.parents:
    if (parent / "src").exists():
        project_root = parent
        break

# Add to PYTHONPATH if not already there
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Now import works
from src.modules import tools_EEG as TEEG

In [ ]:
# 0.2 Load FILES

In [3]:
files = Path("/home/tperezsanchez/FoundationModel_EEG_Dissertation/Main_project/results/XB47Y_28032026Normalized/")
base_files = sorted(files.glob("*full.npz"))

print(f"Found {len(base_files)} .npz files")

rows = []
from datetime import datetime, timezone
import numpy as np


def parse_timestamp(val):
    """Accept Unix float, datetime string, or repeated timestamp arrays."""

    if isinstance(val, np.ndarray):
        flat = val.ravel()
        cleaned = [str(x).strip() for x in flat if str(x).strip() != ""]

        if len(cleaned) == 0:
            return None

        unique_vals = list(dict.fromkeys(cleaned))

        if len(unique_vals) == 1:
            val = unique_vals[0]
        else:
            raise ValueError(f"Timestamp array has multiple different values: {unique_vals}")

    try:
        return float(val)
    except (ValueError, TypeError):
        pass

    val = str(val).strip()
    dt = datetime.strptime(val, "%Y-%m-%d %H:%M:%S.%f")
    return dt.replace(tzinfo=timezone.utc).timestamp() 

Found 183 .npz files


In [4]:
for base_NPZ_path in base_files:
    meta = {"file_name": base_NPZ_path.name, "file_path": str(base_NPZ_path.resolve()), "load_error": None}

    try:
        with np.load(base_NPZ_path, allow_pickle=True) as data:
            keys = set(data.files)
            # store data from key into meta df, convert them into float or str first
            
            meta["fs"]         = float(data["fs"]) if "fs" in keys else None
            meta["source_file"]= str(data["source_file"]) if "source_file" in keys else None
            meta["T0"] = parse_timestamp(data["T0"]) if "T0" in keys else None
            meta["TF"] = parse_timestamp(data["TF"]) if "TF" in keys else None
            meta["is_normalized"] = "mu" in keys and "sigma" in keys

            meta["channel_names"]  = list(data["channel_names"]) if "channel_names" in keys else []
            meta["seizure_onsets"] = list(data["seizure_onsets"]) if "seizure_onsets" in keys else []

            # Read X shape only — avoids loading the full array into memory
            if "X" in keys:
                shape = data["X"].shape
                meta["n_channels"] = int(shape[0]) if len(shape) == 2 else None
                meta["n_samples"]  = int(shape[1]) if len(shape) == 2 else None
            else:
                meta["n_channels"] = None
                meta["n_samples"]  = None

    except Exception as e:
        meta["load_error"] = str(e)
        print(f"Failed to load {base_NPZ_path.name}: {e}")

    rows.append(meta)

print(f"\nLoaded metadata from {len(rows)} file(s)")


Loaded metadata from 183 file(s)


In [9]:
from datetime import datetime

for meta in rows:
    T0 = meta.get("T0")
    TF = meta.get("TF") 

    meta["start_time"] = datetime.fromtimestamp(T0) if T0 is not None else None
    meta["end_time"]   = datetime.fromtimestamp(TF) if TF is not None else None
    meta["duration_s"] = round(TF - T0, 3) if (T0 is not None and TF is not None) else None

    # Cross-check: does n_samples / fs match TF - T0?
    fs, n = meta.get("fs"), meta.get("n_samples")
    if fs and n and meta["duration_s"] is not None:
        expected = round(n / fs, 3)
        meta["duration_check_ok"] = abs(expected - meta["duration_s"]) < 1.0
    else:
        meta["duration_check_ok"] = None

bad = [m for m in rows if m["duration_check_ok"] is False]
print(len(bad), "files with duration mismatch")

0 files with duration mismatch


In [10]:
# 3. General Dataframe for every file
COLUMNS = [
    "file_name", "file_path",
    "start_time", "end_time", "duration_s",
    "fs", "n_channels", "n_samples",
    "channel_names", "seizure_onsets",
    "is_normalized", "source_file",
    "duration_check_ok", "load_error",
]

df = pd.DataFrame(rows, columns=COLUMNS)
df = df.sort_values("start_time", na_position="last").reset_index(drop=True)

print(df.shape)
df.head()

print("── Sanity Report ──────────────────────────────────────")
print(f"  Total recordings   : {len(df)}")
print(f"  Load errors        : {df['load_error'].notna().sum()}")
print(f"  Missing T0/TF      : {df['start_time'].isna().sum()}")
print(f"  Duration check ✗   : {(df['duration_check_ok'] == False).sum()}")
print(f"  Missing fs         : {df['fs'].isna().sum()}")
print(f"  Normalized files   : {df['is_normalized'].sum()}")

# Check for overlapping recordings
valid = df.dropna(subset=["start_time", "end_time"]).sort_values("start_time")
overlaps = 0
for i in range(len(valid) - 1):
    if valid.iloc[i]["end_time"] > valid.iloc[i + 1]["start_time"]:
        overlaps += 1
print(f"  Overlapping pairs  : {overlaps}")
print("───────────────────────────────────────────────────────")
#==========================
#==========================
#==========================
# 4. Clean onset: solution to problems with "nan"
def clean_onsets(x):
    if isinstance(x, (list, np.ndarray)):
        return [i for i in x if not pd.isna(i)]
    elif pd.isna(x):
        return []
    else:
        return [x]

df["seizure_onsets_clean"] = df["seizure_onsets"].apply(clean_onsets)

(183, 14)
── Sanity Report ──────────────────────────────────────
  Total recordings   : 183
  Load errors        : 0
  Missing T0/TF      : 0
  Duration check ✗   : 0
  Missing fs         : 0
  Normalized files   : 183
  Overlapping pairs  : 75
───────────────────────────────────────────────────────


In [11]:
df.head()

,file_name,file_path,start_time,end_time,duration_s,fs,n_channels,n_samples,channel_names,seizure_onsets,is_normalized,source_file,duration_check_ok,load_error,seizure_onsets_clean
0,XB47Y_35_preproc_full.npz,/home/tperezsanchez/FoundationModel_EEG_Disser...,2019-10-29 09:31:04,2019-10-29 10:01:00.730450,1796.730,207.031055,2,371979,"[EEG SQ_D-SQ_C, EEG SQ_P-SQ_C]",[nan],True,['XB47Y_35.mat'],True,None,[]
1,XB47Y_37_preproc_full.npz,/home/tperezsanchez/FoundationModel_EEG_Disser...,2019-10-29 19:54:13,2019-10-30 01:54:12.759550,21599.760,207.031055,2,4471821,"[EEG SQ_D-SQ_C, EEG SQ_P-SQ_C]",[nan],True,['XB47Y_37.mat'],True,None,[]
2,XB47Y_38_preproc_full.npz,/home/tperezsanchez/FoundationModel_EEG_Disser...,2019-10-30 01:54:13,2019-10-30 04:58:50.338150,11077.338,207.031055,2,2293353,"[EEG SQ_D-SQ_C, EEG SQ_P-SQ_C]",[nan],True,['XB47Y_38.mat'],True,None,[]
3,XB47Y_98_preproc_full.npz,/home/tperezsanchez/FoundationModel_EEG_Disser...,2019-10-30 07:47:44,2019-10-30 13:47:44.759400,21600.759,207.031055,2,4472028,"[EEG SQ_D-SQ_C, EEG SQ_P-SQ_C]",[nan],True,['XB47Y_98.mat'],True,None,[]
4,XB47Y_99_preproc_full.npz,/home/tperezsanchez/FoundationModel_EEG_Disser...,2019-10-30 13:47:44,2019-10-30 19:47:44.759400,21600.759,207.031055,2,4472028,"[EEG SQ_D-SQ_C, EEG SQ_P-SQ_C]",[nan],True,['XB47Y_99.mat'],True,None,[]


NameError: name 'rows_windows' is not defined

In [13]:
# 5. Windowing
# Create an empty list that will store one dictionary per EEG window
df_windows = TEEG.create_eeg_windows_2_3(df, window_sec=10)
# Convert the list of dictionaries into a new dataframe
# Each row now represents one 10-second window


print(df_windows.head())
print(df_windows.shape) # rows vs. columns
df_windows[df_windows["seizure_onsets"].apply(lambda x: not pd.isna(x).all() if isinstance(x, (list, np.ndarray)) else not pd.isna(x))].head(10)

                   file_name  window_id  start_sample  end_sample          fs  \
0  XB47Y_35_preproc_full.npz          0             0        2070  207.031055   
1  XB47Y_35_preproc_full.npz          1          2070        4140  207.031055   
2  XB47Y_35_preproc_full.npz          2          4140        6210  207.031055   
3  XB47Y_35_preproc_full.npz          3          6210        8280  207.031055   
4  XB47Y_35_preproc_full.npz          4          8280       10350  207.031055   

   n_channels  window_sec seizure_onsets  
0           2          10             []  
1           2          10             []  
2           2          10             []  
3           2          10             []  
4           2          10             []  
(291113, 8)


,file_name,window_id,start_sample,end_sample,fs,n_channels,window_sec,seizure_onsets
15409,XB47Y_41_preproc_full.npz,0,0,2070,207.031055,2,10,[2019-10-31 23:25:08.153000]
15410,XB47Y_41_preproc_full.npz,1,2070,4140,207.031055,2,10,[2019-10-31 23:25:08.153000]
15411,XB47Y_41_preproc_full.npz,2,4140,6210,207.031055,2,10,[2019-10-31 23:25:08.153000]
15412,XB47Y_41_preproc_full.npz,3,6210,8280,207.031055,2,10,[2019-10-31 23:25:08.153000]
15413,XB47Y_41_preproc_full.npz,4,8280,10350,207.031055,2,10,[2019-10-31 23:25:08.153000]
15414,XB47Y_41_preproc_full.npz,5,10350,12420,207.031055,2,10,[2019-10-31 23:25:08.153000]
15415,XB47Y_41_preproc_full.npz,6,12420,14490,207.031055,2,10,[2019-10-31 23:25:08.153000]
15416,XB47Y_41_preproc_full.npz,7,14490,16560,207.031055,2,10,[2019-10-31 23:25:08.153000]
15417,XB47Y_41_preproc_full.npz,8,16560,18630,207.031055,2,10,[2019-10-31 23:25:08.153000]
15418,XB47Y_41_preproc_full.npz,9,18630,20700,207.031055,2,10,[2019-10-31 23:25:08.153000]


In [15]:
# Labeling:
# step by step
import pandas as pd
import numpy as np

# 1. Helper functions
LABEL_MAP = {
    "interictal": 0,
    "preictal": 1,
    "seizure": 2
}


def overlaps(a_start, a_end, b_start, b_end):
    """
    Return True if two time intervals overlap.
    """
    return (a_start < b_end) and (a_end > b_start)
#window:   |----------|
#seizure:       |----------|
# returns True in that case, because it overlaps

def is_in_gap(window_start, window_end, preictal_end, seizure_start):
    """
    Return True if the window overlaps the gap between
    the preictal interval and the seizure interval.
    detects if a window is in between the space of preictal ending and seizure start
    """
    return overlaps(window_start, window_end, preictal_end, seizure_start)
#onset -10 min    onset -5 min        onset        onset +5 min
#|--- preictal ---|---- gap ----|--- seizure ---|

def get_seizure_intervals(onset, preictal_range_min, ictal_range_min):
    """
    Given one seizure onset, create the preictal and seizure intervals.
    E.g.:
    onset = "2026-04-27 12:00:00"
    preictal_range_min = (-10, -5)
    ictal_range_min = (0, 5)
    returns:
    preictal: 11:50 to 11:55
    seizure:  12:00 to 12:05
    """
    onset = pd.to_datetime(onset)

    preictal_start = onset + pd.Timedelta(minutes=preictal_range_min[0])
    preictal_end = onset + pd.Timedelta(minutes=preictal_range_min[1])

    seizure_start = onset + pd.Timedelta(minutes=ictal_range_min[0])
    seizure_end = onset + pd.Timedelta(minutes=ictal_range_min[1])

    return preictal_start, preictal_end, seizure_start, seizure_end
# 2. Prepare dataframe
def initialize_labeled_dataframe(df_windows):
    """
    Create a copy of df_windows and add empty columns
    needed for labeling.
    df_windows original
        ↓
    initialize_labeled_dataframe()
        ↓
    df_labeled with empty columns
    """
    # 2.1 creates a copy to preserve original dataframe
    df_labeled = df_windows.copy()
    # 2.2 add new columns
    df_labeled["window_start_time"] = pd.NaT
    df_labeled["window_end_time"] = pd.NaT
    df_labeled["class_label"] = np.nan
    df_labeled["label_name"] = pd.NA
    df_labeled["excluded_reason"] = pd.NA

    return df_labeled
# 3. Calculate real time per window:
def compute_window_times(row, recording_start):
    """
    Compute the real datetime start and end of one EEG window.

    Parameters
    ----------
    row : pd.Series
        One row from df_windows.

    recording_start : datetime-like
        Start time of the recording.

    Returns
    -------
    window_start_time : pd.Timestamp
    window_end_time : pd.Timestamp
    -----
    start_sample / fs = seconds from beginning of recording
    end_sample / fs   = seconds from end of recording

    E.g:
    start_sample = 2070
    end_sample = 4140
    fs = 207
    
    start_sec = 10 segundos
    end_sec   = 20 segundos
    if recording started:
    2026-04-27 12:00:00
    then:
    window_start_time = 2026-04-27 12:00:10
    window_end_time   = 2026-04-27 12:00:20
    """

    recording_start = pd.to_datetime(recording_start)

    fs = row["fs"]

    start_sec = row["start_sample"] / fs
    end_sec = row["end_sample"] / fs

    window_start_time = recording_start + pd.Timedelta(seconds=start_sec)
    window_end_time = recording_start + pd.Timedelta(seconds=end_sec)

    return window_start_time, window_end_time
# 4. Search for corresponding recording
def get_matching_recording(row, df_recordings):
    """
    Find the recording-level metadata corresponding to one window.
        row["file_name"]
            ↓
    search for file name in df_recording
            ↓
    returns corresponding row
    """

    file_name = row["file_name"]

    matching_rec = df_recordings[
        df_recordings["file_name"] == file_name
    ]

    if matching_rec.empty:
        raise ValueError(
            f"No matching recording found in df_recordings for file_name: {file_name}"
        )

    rec = matching_rec.iloc[0]

    return rec
# 5. get real time per window
def get_window_datetime_info(row, df_recordings):
    """
    Get the real start and end datetime of one EEG window.
        row de df_windows
            ↓
    search for metadata in df_recordings
            ↓
    extract start_time from recording
            ↓
    convert start_sample/end_sample into real datetime
    """

    rec = get_matching_recording(row, df_recordings)

    window_start_time, window_end_time = compute_window_times(
        row=row,
        recording_start=rec["start_time"]
    )

    return window_start_time, window_end_time
# 6. Labels one window at a time
def label_single_window(
    window_start_time,
    window_end_time,
    seizure_onsets,
    preictal_range_min=(-10, -5),
    ictal_range_min=(0, 5),
    include_gap_as_interictal=True
):
    """
    Label one EEG window as interictal, preictal, seizure,
    or mark it for exclusion if it falls in the periictal gap.
    
    One window + seizure_onsets
            ↓
    overlap with seizure?
            ↓
    yes → seizure
    
    elif:
            ↓
    overlap with preictal?
            ↓
    yes → preictal
    
    else:
            ↓
    ¿overlap with gap?
            ↓
    yes and include_gap_as_interictal=False → exclude
    
    if nothing:
            ↓
    interictal
    """

    # Default label
    assigned_label = LABEL_MAP["interictal"]
    assigned_name = "interictal"
    excluded_reason = pd.NA

    # Clean seizure onsets
    seizure_onsets = clean_onsets(seizure_onsets)

    # If no seizure onsets exist, keep interictal
    if len(seizure_onsets) == 0:
        return assigned_label, assigned_name, excluded_reason

    overlaps_gap = False

    for onset in seizure_onsets:

        preictal_start, preictal_end, seizure_start, seizure_end = (
            get_seizure_intervals(
                onset=onset,
                preictal_range_min=preictal_range_min,
                ictal_range_min=ictal_range_min
            )
        )

        # 1. Seizure has highest priority
        if overlaps(
            window_start_time,
            window_end_time,
            seizure_start,
            seizure_end
        ):
            assigned_label = LABEL_MAP["seizure"]
            assigned_name = "seizure"
            excluded_reason = pd.NA
            break

        # 2. Preictal has second priority
        elif overlaps(
            window_start_time,
            window_end_time,
            preictal_start,
            preictal_end
        ):
            assigned_label = LABEL_MAP["preictal"]
            assigned_name = "preictal"
            excluded_reason = pd.NA

        # 3. Optional gap detection
        elif preictal_end < seizure_start:
            if is_in_gap(
                window_start_time,
                window_end_time,
                preictal_end,
                seizure_start
            ):
                overlaps_gap = True

    # Exclude gap windows if requested
    if overlaps_gap and not include_gap_as_interictal:
        assigned_label = np.nan
        assigned_name = pd.NA
        excluded_reason = "periictal_gap"

    return assigned_label, assigned_name, excluded_reason
# 7. loop through all windows
def apply_window_labeling(
    df_labeled,
    df_recordings,
    preictal_range_min=(-10, -5),
    ictal_range_min=(0, 5),
    include_gap_as_interictal=True
):
    """
    Apply window labeling row by row to the full dataframe.
    """

    for idx, row in df_labeled.iterrows():

        # 1. Compute real datetime of the window
        window_start_time, window_end_time = get_window_datetime_info(
            row=row,
            df_recordings=df_recordings
        )

        # 2. Label one window
        assigned_label, assigned_name, excluded_reason = label_single_window(
            window_start_time=window_start_time,
            window_end_time=window_end_time,
            seizure_onsets=row["seizure_onsets"],
            preictal_range_min=preictal_range_min,
            ictal_range_min=ictal_range_min,
            include_gap_as_interictal=include_gap_as_interictal
        )

        # 3. Save results
        df_labeled.at[idx, "window_start_time"] = window_start_time
        df_labeled.at[idx, "window_end_time"] = window_end_time
        df_labeled.at[idx, "class_label"] = assigned_label
        df_labeled.at[idx, "label_name"] = assigned_name
        df_labeled.at[idx, "excluded_reason"] = excluded_reason

    return df_labeled


In [19]:
df_labeled = initialize_labeled_dataframe(df_windows)

df_labeled = apply_window_labeling(
    df_labeled=df_labeled,
    df_recordings=df,
    preictal_range_min=(-10, -5),
    ictal_range_min=(0, 5),
    include_gap_as_interictal=True
)

In [28]:
df_labeled.head()

,file_name,window_id,start_sample,end_sample,fs,n_channels,window_sec,seizure_onsets,window_start_time,window_end_time,class_label,label_name,excluded_reason
0,XB47Y_35_preproc_full.npz,0,0,2070,207.031055,2,10,[],2019-10-29 09:31:04.000000,2019-10-29 09:31:13.998500,0.0,interictal,<NA>
1,XB47Y_35_preproc_full.npz,1,2070,4140,207.031055,2,10,[],2019-10-29 09:31:13.998500,2019-10-29 09:31:23.997000,0.0,interictal,<NA>
2,XB47Y_35_preproc_full.npz,2,4140,6210,207.031055,2,10,[],2019-10-29 09:31:23.997000,2019-10-29 09:31:33.995500,0.0,interictal,<NA>
3,XB47Y_35_preproc_full.npz,3,6210,8280,207.031055,2,10,[],2019-10-29 09:31:33.995500,2019-10-29 09:31:43.994000,0.0,interictal,<NA>
4,XB47Y_35_preproc_full.npz,4,8280,10350,207.031055,2,10,[],2019-10-29 09:31:43.994000,2019-10-29 09:31:53.992500,0.0,interictal,<NA>


In [29]:
# -------------------------------
# Keep only preictal and seizure windows
# -------------------------------
df_final = df_labeled[df_labeled["class_label"].isin([1, 2])].copy()

# Make class_label integer
df_final["class_label"] = df_final["class_label"].astype(int)

In [30]:
df_final.head()
df_final.shape
df_final.columns
df_final.info()
df_final.describe()

<class 'pandas.core.frame.DataFrame'>
Index: 2149 entries, 16663 to 290876
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   file_name          2149 non-null   object        
 1   window_id          2149 non-null   int64         
 2   start_sample       2149 non-null   int64         
 3   end_sample         2149 non-null   int64         
 4   fs                 2149 non-null   float64       
 5   n_channels         2149 non-null   int64         
 6   window_sec         2149 non-null   int64         
 7   seizure_onsets     2149 non-null   object        
 8   window_start_time  2149 non-null   datetime64[ns]
 9   window_end_time    2149 non-null   datetime64[ns]
 10  class_label        2149 non-null   int64         
 11  label_name         2149 non-null   object        
 12  excluded_reason    0 non-null      object        
dtypes: datetime64[ns](2), float64(1), int64(6), object(4)
memory u

,window_id,start_sample,end_sample,fs,n_channels,window_sec,window_start_time,window_end_time,class_label
count,2149.000000,2.149000e+03,2.149000e+03,2.149000e+03,2149.0,2149.0,2149,2149,2149.000000
mean,1024.024197,2.119730e+06,2.121800e+06,2.070311e+02,2.0,10.0,2019-11-18 00:49:00.742697472,2019-11-18 00:49:10.741197568,1.512797
min,9.000000,1.863000e+04,2.070000e+04,2.070311e+02,2.0,10.0,2019-10-31 23:15:03.118999591,2019-10-31 23:15:13.117499590,1.000000
25%,459.000000,9.501300e+05,9.522000e+05,2.070311e+02,2.0,10.0,2019-11-08 21:46:54.580000,2019-11-08 21:47:04.578499840,1.000000
50%,1084.000000,2.243880e+06,2.245950e+06,2.070311e+02,2.0,10.0,2019-11-12 23:17:04.992999680,2019-11-12 23:17:14.991499520,2.000000
75%,1460.000000,3.022200e+06,3.024270e+06,2.070311e+02,2.0,10.0,2019-11-30 20:50:33.514999552,2019-11-30 20:50:43.513499648,2.000000
max,2152.000000,4.454640e+06,4.456710e+06,2.070311e+02,2.0,10.0,2019-12-11 19:27:07.144499596,2019-12-11 19:27:17.142999596,2.000000
std,579.155237,1.198851e+06,1.198851e+06,8.158388e-10,0.0,0.0,NaN,NaN,0.499953


In [ ]:
from datetime import timedelta
import pandas as pd
import numpy as np


def label_eeg_windows(
    df_windows,
    df_recordings,
    preictal_range_min=(-10, -5),
    ictal_range_min=(0, 5),
    include_gap_as_interictal=True,
    keep_only_preictal_seizure=False,
    datetime_as_string=False,
    print_summary=True
):
    """
    Label EEG windows as interictal, preictal, or seizure.

    Label mapping:
        interictal = 0
        preictal   = 1
        seizure    = 2

    Parameters
    ----------
    df_windows : pd.DataFrame
        DataFrame with one row per EEG window.
        Expected columns:
            - file_name
            - window_id
            - start_sample
            - end_sample
            - fs
            - seizure_onsets

    df_recordings : pd.DataFrame
        Recording-level metadata DataFrame.
        Expected columns:
            - file_name
            - start_time

    preictal_range_min : tuple
        Time range before seizure onset for preictal labeling, in minutes.
        Example:
            (-10, -5) means from 10 minutes before onset to 5 minutes before onset.

    ictal_range_min : tuple
        Time range around/after seizure onset for seizure labeling, in minutes.
        Example:
            (0, 5) means from seizure onset to 5 minutes after onset.

    include_gap_as_interictal : bool
        If True, windows that are close to a seizure but do not overlap with
        preictal or ictal intervals are labeled as interictal.

        If False, windows between the preictal interval and ictal interval
        are excluded.

        Example with preictal_range_min=(-10, -5) and ictal_range_min=(0, 5):
            onset - 4 min to onset - 1 min is a "gap".
            If True  -> interictal
            If False -> excluded

    keep_only_preictal_seizure : bool
        If True, return only preictal and seizure windows.
        If False, return all labeled windows, including interictal.

    datetime_as_string : bool
        If True, convert window_start_time and window_end_time to strings.
        If False, keep them as pandas datetime objects.

    print_summary : bool
        If True, print label mapping and class counts.

    Returns
    -------
    df_labeled : pd.DataFrame
        DataFrame with labeled EEG windows.
    """

    # -------------------------------
    # Label mapping
    # -------------------------------
    label_map = {
        "interictal": 0,
        "preictal": 1,
        "seizure": 2
    }

    # -------------------------------
    # Helper: overlap function
    # -------------------------------
    def overlaps(a_start, a_end, b_start, b_end):
        """
        Return True if two time intervals overlap.
        """
        return (a_start < b_end) and (a_end > b_start)

    # -------------------------------
    # Helper: check if window is inside the gap between preictal and ictal
    # -------------------------------
    def is_in_gap(window_start, window_end, preictal_end, ictal_start):
        """
        Return True if a window overlaps the gap between the end of the preictal
        interval and the beginning of the ictal interval.
        """
        return overlaps(window_start, window_end, preictal_end, ictal_start)

    # -------------------------------
    # Copy dataframe to avoid overwriting original
    # -------------------------------
    df_labeled = df_windows.copy()

    # Create new columns
    df_labeled["window_start_time"] = pd.NaT
    df_labeled["window_end_time"] = pd.NaT
    df_labeled["class_label"] = np.nan
    df_labeled["label_name"] = pd.NA

    # Optional column to track excluded gap windows
    df_labeled["excluded_reason"] = pd.NA

    # -------------------------------
    # Main labeling loop
    # -------------------------------
    for idx, row in df_labeled.iterrows():

        # Match recording-level metadata
        matching_rec = df_recordings[df_recordings["file_name"] == row["file_name"]]

        if matching_rec.empty:
            raise ValueError(
                f"No matching recording found in df_recordings for file_name: {row['file_name']}"
            )

        rec = matching_rec.iloc[0]

        # Recording start time
        recording_start = pd.to_datetime(rec["start_time"])
        fs = row["fs"]

        # Convert sample indices to seconds
        start_sec = row["start_sample"] / fs
        end_sec = row["end_sample"] / fs

        # Compute real datetime of each window
        window_start_time = recording_start + pd.Timedelta(seconds=start_sec)
        window_end_time = recording_start + pd.Timedelta(seconds=end_sec)

        # Save window datetime information
        df_labeled.at[idx, "window_start_time"] = window_start_time
        df_labeled.at[idx, "window_end_time"] = window_end_time

        # Default label = interictal
        assigned_label = label_map["interictal"]
        assigned_name = "interictal"
        excluded_reason = pd.NA

        # Clean seizure_onsets
        seizure_onsets = clean_onsets(row["seizure_onsets"])

        # If there are no seizure onsets, keep the default label: interictal
        if len(seizure_onsets) == 0:
            df_labeled.at[idx, "class_label"] = assigned_label
            df_labeled.at[idx, "label_name"] = assigned_name
            continue

        # Flag used only if include_gap_as_interictal=False
        overlaps_gap = False

        # Loop through every seizure onset associated with this window/recording
        for onset in seizure_onsets:

            onset = pd.to_datetime(onset)

            # Define preictal interval
            preictal_start = onset + pd.Timedelta(minutes=preictal_range_min[0])
            preictal_end = onset + pd.Timedelta(minutes=preictal_range_min[1])

            # Define ictal/seizure interval
            seizure_start = onset + pd.Timedelta(minutes=ictal_range_min[0])
            seizure_end = onset + pd.Timedelta(minutes=ictal_range_min[1])

            # -------------------------------
            # 1) Seizure has highest priority
            # -------------------------------
            if overlaps(window_start_time, window_end_time, seizure_start, seizure_end):
                assigned_label = label_map["seizure"]
                assigned_name = "seizure"
                excluded_reason = pd.NA
                break

            # -------------------------------
            # 2) Preictal has second priority
            # -------------------------------
            elif overlaps(window_start_time, window_end_time, preictal_start, preictal_end):
                assigned_label = label_map["preictal"]
                assigned_name = "preictal"
                excluded_reason = pd.NA

            # -------------------------------
            # 3) Optional: detect gap between preictal and seizure
            # -------------------------------
            elif preictal_end < seizure_start:
                if is_in_gap(window_start_time, window_end_time, preictal_end, seizure_start):
                    overlaps_gap = True

        # -------------------------------
        # Exclude gap windows if requested
        # -------------------------------
        if overlaps_gap and not include_gap_as_interictal:
            assigned_label = np.nan
            assigned_name = pd.NA
            excluded_reason = "periictal_gap"

        # Save final assigned label
        df_labeled.at[idx, "class_label"] = assigned_label
        df_labeled.at[idx, "label_name"] = assigned_name
        df_labeled.at[idx, "excluded_reason"] = excluded_reason

    # -------------------------------
    # Remove excluded windows, if any
    # -------------------------------
    df_labeled = df_labeled[df_labeled["class_label"].notna()].copy()

    # -------------------------------
    # Optional: keep only preictal and seizure windows
    # -------------------------------
    if keep_only_preictal_seizure:
        df_labeled = df_labeled[
            df_labeled["label_name"].isin(["preictal", "seizure"])
        ].copy()

    # Make class_label integer
    df_labeled["class_label"] = df_labeled["class_label"].astype(int)

    # -------------------------------
    # Optional: convert datetime columns to string format
    # -------------------------------
    if datetime_as_string:
        df_labeled["window_start_time"] = pd.to_datetime(
            df_labeled["window_start_time"]
        ).dt.strftime("%Y-%m-%d %H:%M:%S")

        df_labeled["window_end_time"] = pd.to_datetime(
            df_labeled["window_end_time"]
        ).dt.strftime("%Y-%m-%d %H:%M:%S")

    # -------------------------------
    # Print summary
    # -------------------------------
    if print_summary:
        print("Label mapping:")
        print("  0 = interictal")
        print("  1 = preictal")
        print("  2 = seizure")

        print("\nClass counts:")
        print(df_labeled["label_name"].value_counts(dropna=False))

        print("\nClass counts by numeric label:")
        print(df_labeled["class_label"].value_counts().sort_index())

    return df_labeled